# 05 · Deep-learning inputs

Step 2 of the [Phase 2 protocol](../docs/development_protocol.md): turn the Phase 1 features into network inputs without changing what the model may know.

* **Same rows, same features** as Phase 1: everything comes from `aerocast.pipeline.prepare()`.
* **Train-only statistics:** category vocabularies, the rare bucket, imputation medians and the scaler are fitted on the rows the network is fitted on (per CV fold, or the final training rows).
* **Fitting clock:** the final model is fitted for 2013-09-01, so training rows whose label is not yet known at 20:00 on 31 Aug are excluded.

All logic lives in `aerocast_dl`; this notebook only calls it and checks the saved results.

In [1]:
import sys, json, pathlib
ROOT = pathlib.Path.cwd().resolve().parent
for p in (ROOT / 'src', ROOT.parent / 'phase1' / 'src'):
    sys.path.insert(0, str(p))

import aerocast_dl  # torch before XGBoost (OpenMP)
import numpy as np, pandas as pd
from aerocast import pipeline as p1_pipeline
from aerocast_dl import config, tensors

P = p1_pipeline.prepare()
P.split_summary()

{'train': {'rows': 211684,
  'delay_rate': 0.26489484325693013,
  'from': '2013-01-08',
  'to': '2013-08-31'},
 'calib': {'rows': 55628,
  'delay_rate': 0.15695333285395843,
  'from': '2013-09-01',
  'to': '2013-10-31'},
 'test': {'rows': 53991,
  'delay_rate': 0.2583023096442,
  'from': '2013-11-01',
  'to': '2013-12-31'}}

## 1. Fitting clock

Only training rows with labels available at the decision clock for 2013-09-01 may fit the network or its encoder. `fit_encoder` raises `LeakageError` for any other row.

In [2]:
rows = tensors.final_fit_rows(P)
summary = json.loads((config.RESULTS_DIR / 'data_summary.json').read_text())
assert summary['fit_rows'] == len(rows)
{k: summary[k] for k in ['fit_date', 'train_rows', 'fit_rows', 'fit_rows_excluded_by_clock']}

{'fit_date': '2013-09-01',
 'train_rows': 211684,
 'fit_rows': 211517,
 'fit_rows_excluded_by_clock': 167}

## 2. Inputs

Categorical inputs map to embedding indices; index 0 is shared by rare (< 50 fitting rows, the Phase 1 one-hot `min_frequency`), unseen and missing values. Numeric inputs get Phase 1's preprocessing: median imputation with missing indicators, then standardisation.

In [3]:
enc = tensors.fit_encoder(P, rows, config.FINAL_FIT_DATE, ids=config.ID_COLS)
cats, nums = tensors.model_inputs(P, config.ID_COLS)
E = enc.transform(cats, nums)
assert dict(zip(E.cat_cols, E.cardinalities)) == summary['embedding_tables']
assert len(E.num_cols) == summary['numeric_inputs_with_missing_indicators']
print('rows:', len(E), '| numeric inputs:', len(E.num_cols), '| embedding tables:', summary['embedding_tables'])
print('missing indicators:', summary['missing_indicator_columns'])

rows: 321303 | numeric inputs: 40 | embedding tables: {'carrier': 16, 'origin': 4, 'dest': 94, 'tailnum': 1569, 'carrier_flight': 1394, 'origin_hour': 55}
missing indicators: []


## 3. Cold start

Share of rows whose category falls in the shared index 0 of a vocabulary fitted on the final training rows. This uses category membership only, never labels. Carrier, origin and destination are almost always embedded; tail numbers and especially carrier-flight numbers are often rare or new after August, which is why the high-cardinality IDs are an ablation, not part of the default model.

In [4]:
cold = tensors.cold_start_table(P)
saved = pd.read_csv(config.RESULTS_DIR / 'cold_start.csv')
pd.testing.assert_frame_equal(cold, saved, check_dtype=False)
cold.pivot(index='column', columns='split', values='unk_share').loc[config.CAT_COLS + config.ID_COLS].round(3)

split,calib,test,train
column,,,
carrier,0.000,0.000,0.000
origin,0.000,0.000,0.000
dest,0.001,0.002,0.001
tailnum,0.239,0.266,0.219
carrier_flight,0.353,0.439,0.203
origin_hour,0.000,0.001,0.000


## 4. Leakage checks for this step

The tests in `tests/test_tensors.py` assert, on the real data:

| Test | Asserts |
|---|---|
| `test_tensor_alignment` | encoded rows equal the Phase 1 rows, labels and values 1:1 |
| `test_fit_stats_train_only` | scrambling every Sep–Dec row leaves vocabularies, medians and scaler unchanged |
| `test_fit_encoder_rejects_forbidden_rows` | calibration rows and labels unknown at the clock raise `LeakageError` |
| `test_no_missing_tail_in_target` | a blank tail number occurs only for cancelled flights, which the target excludes |
| `test_ids_ignore_outcome_columns` | ID inputs do not change when every post-departure column is scrambled |